# Spatial Joins Exercises

Here\'s a reminder of some of the functions we have seen. Hint: they
should be useful for the exercises!

-   `sum(expression)`: aggregate to
    return a sum for a set of records
-   `count(expression)`: aggregate to
    return the size of a set of records
-   `ST_Area(geometry)` returns the
    area of the polygons
-   `ST_AsText(geometry)` returns WKT `text`
-   `ST_Contains(geometry A, geometry B)` returns the true if geometry A contains geometry B
-   `ST_Distance(geometry A, geometry B)` returns the minimum distance between geometry A and
    geometry B
-   `ST_DWithin(geometry A, geometry B, radius)` returns the true if geometry A is radius distance or less from geometry B
-   `ST_GeomFromText(text)` returns `geometry`
-   `ST_Intersects(geometry A, geometry B)` returns the true if geometry A intersects geometry B
-   `ST_Length(linestring)` returns the length of the linestring
-   `ST_Touches(geometry A, geometry B)` returns the true if the boundary of geometry A touches geometry B
-   `ST_Within(geometry A, geometry B)` returns the true if geometry A is within geometry B


Uncomment and run the following cell to install the required packages.


In [2]:
%pip install duckdb leafmap lonboard
import duckdb
import leafmap


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install duckdb duckdb-engine jupysql


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to ereate SQl cells
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb:///:memory:


Download the [nyc_data.zip](https://github.com/opengeos/data/raw/main/duckdb/nyc_data.zip) dataset using leafmap. The zip file contains the following datasets. Create a new DuckDB database and import the datasets into the database. Each dataset should be imported into a separate table.

- nyc_census_blocks
- nyc_homicides
- nyc_neighborhoods
- nyc_streets
- nyc_subway_stations

In [5]:
# Check your working directory
import os
os.getcwd()

'/Users/wangentao/Documents/CASA/bd'

1. **What subway station is in \'Little Italy\'? What subway route is it on?**

In [3]:
from pathlib import Path
import os

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

url = "https://github.com/opengeos/data/raw/main/duckdb/nyc_data.zip"

leafmap.download_file(
    url,
    output=str(DATA_DIR / "nyc_data.zip"),
    unzip=True,
    overwrite=True
)

Downloading...
From: https://github.com/opengeos/data/raw/main/duckdb/nyc_data.zip
To: /Users/wangentao/Documents/CASA/bd/data/nyc_data.db.zip
100%|██████████| 8.73M/8.73M [00:01<00:00, 4.51MB/s]


Extracting files...


'/Users/wangentao/Documents/CASA/bd/data/nyc_data.db.zip'

In [6]:
%%sql
LOAD spatial;

CREATE TABLE IF NOT EXISTS nyc_subway_stations AS
SELECT * FROM ST_Read('data/nyc_subway_stations.shp');

CREATE TABLE IF NOT EXISTS nyc_neighborhoods AS
SELECT * FROM ST_Read('data/nyc_neighborhoods.shp');

CREATE TABLE IF NOT EXISTS nyc_streets AS
SELECT * FROM ST_Read('data/nyc_streets.shp');

CREATE TABLE IF NOT EXISTS nyc_homicides AS
SELECT * FROM ST_Read('data/nyc_homicides.shp');

CREATE TABLE IF NOT EXISTS nyc_census_blocks AS
SELECT * FROM ST_Read('data/nyc_census_blocks.shp');



,Success


In [7]:
%%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name;

,table_name
0,nyc_census_blocks
1,nyc_homicides
2,nyc_neighborhoods
3,nyc_streets
4,nyc_subway_stations


In [9]:
%%sql
SELECT s.name, s.routes
FROM nyc_subway_stations AS s
JOIN nyc_neighborhoods AS n
ON ST_Contains(n.geom, s.geom)
WHERE n.name = 'Little Italy';


,NAME,ROUTES
0,Spring St,6


2. **What are all the neighborhoods served by the 6-train?** (Hint: The `routes` column in the `nyc_subway_stations` table has values like \'B,D,6,V\' and \'C,6\')


In [13]:
%%sql
SELECT DISTINCT n.name, n.boroname
FROM nyc_neighborhoods AS n
JOIN nyc_subway_stations AS s
  ON ST_Intersects(n.geom, s.geom)
WHERE s.routes LIKE '%6%'
ORDER BY n.name;

,NAME,BORONAME
0,Chinatown,Manhattan
1,East Harlem,Manhattan
2,Financial District,Manhattan
3,Gramercy,Manhattan
4,Greenwich Village,Manhattan
5,Hunts Point,The Bronx
6,Little Italy,Manhattan
7,Midtown,Manhattan
8,Mott Haven,The Bronx
9,Murray Hill,Manhattan


3. **After 9/11, the \'Battery Park\' neighborhood was off limits for several days. How many people had to be evacuated?**

In [14]:
%%sql
SELECT SUM(b.popn_total) AS total_evacuated
FROM nyc_census_blocks AS b
JOIN nyc_neighborhoods AS n
  ON ST_Intersects(n.geom, b.geom)
WHERE n.name = 'Battery Park';

,total_evacuated
0,17153.0


4. **What neighborhood has the highest population density (persons/km2)?**


In [16]:
%%sql
SELECT 
    n.name, 
    SUM(b.popn_total) / (ST_Area(n.geom) / 1000000.0) AS density
FROM nyc_neighborhoods AS n
JOIN nyc_census_blocks AS b 
  ON ST_Intersects(n.geom, b.geom)
GROUP BY n.name, n.geom
ORDER BY density DESC
LIMIT 5;

,NAME,density
0,North Sutton Area,68435.132838
1,East Village,50404.483413
2,Chinatown,48825.180551
3,Carnegie Hill,48543.725404
4,Upper East Side,48524.487749


When you're finished, you can check your answers [here](https://postgis.net/workshops/postgis-intro/joins_exercises.html).

# Ship-to-Ship Transfer Detection

Now for a less structured exercise. We're going to look at ship-to-ship transfers. The idea is that two ships meet up in the middle of the ocean, and one ship transfers cargo to the other. This is a common way to avoid sanctions, and is often used to transfer oil from sanctioned countries to other countries. We're going to look at a few different ways to detect these transfers using AIS data.

In [ ]:
%pip install duckdb duckdb-engine jupysql

In [ ]:
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to create SQL cells
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb:///:memory:

In [ ]:
%%sql
INSTALL httpfs;
LOAD httpfs;
INSTALL spatial;
LOAD spatial;

## Step 1

Create a spatial database using the following AIS (Automatic Identification System) data:

https://storage.googleapis.com/qm2/casa0025_ships.csv

Each row in this dataset is an AIS 'ping' indicating the position of a ship at a particular date/time, alongside vessel-level characteristics.

It contains the following columns:
* `vesselid`: A unique numerical identifier for each ship, like a license plate
* `vessel_name`: The ship's name
* `vsl_descr`: The ship's type
* `dwt`: The ship's Deadweight Tonnage (how many tons it can carry)
* `v_length`: The ship's length in meters
* `draught`: How many meters deep the ship is draughting (how low it sits in the water). Effectively indicates how much cargo the ship is carrying
* `sog`: Speed over Ground (in knots)
* `date`: A timestamp for the AIS signal
* `lat`: The latitude of the AIS signal (EPSG:4326)
* `lon`: The longitude of the AIS signal (EPSG:4326)

Create a table called 'ais' where each row is a different AIS ping, with no superfluous information. Construct a geometry column.

Create a second table called 'vinfo' which contains vessel-level information with no superfluous information.

You can set a spatial index on each of these tables as follows:

`CREATE INDEX index_name ON table_name USING RTREE(geom);`

In [35]:
%%sql
DROP TABLE ais

,Success


In [37]:
%%sql
CREATE TABLE ais AS
SELECT 
    vesselid, 
    date AS timestamp, 
    sog, 
    draught,
    ST_Point(lon, lat) AS geom  -- 将经纬度转为几何点
FROM 'https://storage.googleapis.com/qm2/casa0025_ships.csv'

,Success


In [38]:
%%sql
SELECT * FROM ais 

,vesselid,timestamp,sog,draught,geom
0,350053,2022-07-25 02:53:29,5.2,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,350053,2022-07-25 03:09:37,0.7,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,350053,2022-07-25 03:13:58,0.7,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,350053,2022-07-25 04:16:06,0.1,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,350053,2022-07-25 05:20:17,0.0,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
...,...,...,...,...,...
101323,217531,2022-08-10 14:16:47,0.1,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
101324,217531,2022-08-10 14:43:48,0.1,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
101325,217531,2022-08-10 15:04:28,5.8,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
101326,217531,2022-08-23 06:06:51,8.3,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [19]:
%%sql
CREATE TABLE vinfo AS
SELECT DISTINCT 
    vesselid, 
    vessel_name, 
    vsl_descr, 
    v_length, 
    dwt
FROM 'https://storage.googleapis.com/qm2/casa0025_ships.csv'

,Success


In [40]:
%%sql
SELECT * FROM vinfo


,vesselid,vessel_name,vsl_descr,v_length,dwt
0,278778,Omskiy-103,general cargo,108.0,3283.0
1,251266,Orel 1,general cargo with container capacity,108.0,3104.0
2,8928830,Panamax Christina,bulk carrier,223.0,82176.0
3,268188,Peter S.,bulk carrier,215.0,71550.0
4,268844,Petra Star,bulk carrier,182.0,43682.0
...,...,...,...,...,...
830,151024,Modus,general cargo,108.0,3111.0
831,305203,Navaho,general cargo with container capacity,139.0,5221.0
832,12974752,Navis-6,general cargo with container capacity,123.0,6316.0
833,9108823,Olimpia,general cargo,127.0,6820.0


In [33]:
%%sql
CREATE INDEX ais_geom_idx ON ais USING RTREE(geom)

,Success


In [34]:
%%sql
SELECT * FROM duckdb_indexes() WHERE index_name = 'ais_geom_idx';

,database_name,database_oid,schema_name,schema_oid,index_name,index_oid,table_name,table_oid,comment,tags,is_unique,is_primary,expressions,sql
0,memory,592,main,590,ais_geom_idx,2477,ais,2471,None,{},False,False,[geom],CREATE INDEX ais_geom_idx ON ais USING RTREE (...


## Step 2

Use a spatial join to identify ship-to-ship transfers in this dataset.
Two ships are considered to be conducting a ship to ship transfer IF:

* They are within 500 meters of each other
* For more than two hours
* And their speed is lower than 1 knot

Some things to consider: make sure you're not joining ships with themselves. Try working with subsets of the data first while you try different things out.

In [39]:
%%sql
SELECT 
    a.vesselid AS ship1_id,
    b.vesselid AS ship2_id,
    MIN(a.timestamp) AS start_time,
    MAX(a.timestamp) AS end_time,
    date_diff('minute', MIN(a.timestamp), MAX(a.timestamp)) AS duration_minutes
FROM ais AS a
JOIN ais AS b 
  ON ST_DWithin(a.geom, b.geom, 0.005) 
  AND abs(date_diff('minute', a.timestamp, b.timestamp)) < 5
WHERE 
  a.vesselid < b.vesselid 
  AND a.sog < 1 
  AND b.sog < 1
GROUP BY a.vesselid, b.vesselid
HAVING duration_minutes > 120;

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ship1_id,ship2_id,start_time,end_time,duration_minutes
0,245060,276883,2022-06-28 17:43:46,2022-06-29 04:45:23,662
1,230124,303701,2022-06-01 15:12:06,2022-06-01 18:33:00,201
2,231063,255247,2022-08-08 17:01:37,2022-08-09 02:49:42,588
3,231063,345658,2022-08-09 17:05:38,2022-08-09 20:23:40,198
4,231063,355557,2022-08-27 20:05:34,2022-08-30 19:43:35,4298
...,...,...,...,...,...
527,296743,357535,2022-07-29 11:49:46,2022-07-30 14:09:21,1580
528,144270,223346,2022-07-18 08:30:14,2022-07-18 11:42:15,192
529,235461,263346,2022-07-18 06:11:05,2022-07-31 04:21:30,18610
530,235461,269668,2022-07-22 23:12:41,2022-07-23 17:45:51,1113
